In [ ]:
# Gọi các thư viện cần thiết 
import numpy as np
import pandas as pd # Xu lý bảng
import seaborn as sns # Vẽ biểu đồ thị của dữ liệu
import matplotlib.pyplot as plt 
from sklearn.preprocessing import StandardScaler # Xử lý chuẩn hóa dữ liệu
from sklearn.model_selection import train_test_split # Chia dữ liệu ra làm 2 phần
from keras.layers import Dense, Activation, Dropout, BatchNormalization, LSTM    # LSTM  biên dạng ANN, BatchNormalization: cho nhỏ lại
from keras.models import Sequential
from tensorflow.keras.utils import to_categorical # Sử dung để làm nổi đối tượng cần phân loại
from keras import callbacks 
from sklearn.metrics import precision_score, recall_score, confusion_matrix, classification_report, accuracy_score, f1_score # Để đo lường

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.utils import np_utils
from tensorflow.keras.preprocessing import image
from tensorflow.keras.optimizers import RMSprop
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np
import cv2
import os

In [ ]:
image_generator = ImageDataGenerator(rescale=1/255, validation_split=0.2)    

train_dataset = image_generator.flow_from_directory(batch_size=32,
                                                 directory='../input/datasetsfacedetect',
                                                 shuffle=True,
                                                 target_size=(150, 150), 
                                                 subset="training",
                                                 class_mode='categorical')

validation_dataset = image_generator.flow_from_directory(batch_size=32,
                                                 directory='../input/datasetsfacedetect',
                                                 shuffle=True,
                                                 target_size=(150, 150), 
                                                 subset="validation",
                                                 class_mode='categorical')

In [ ]:
import glob
true = glob.glob('../input/datasetsfacedetect/True/*.*')
false = glob.glob('../input/datasetsfacedetect/False/*.*')

data = []
labels = []

for i in true:   
    image=tf.keras.preprocessing.image.load_img(i, color_mode='rgb', 
    target_size= (150,150))
    image=np.array(image)
    data.append(image)
    labels.append(1)
for j in false:   
    image=tf.keras.preprocessing.image.load_img(j, color_mode='rgb', 
    target_size= (150,150))
    image=np.array(image)
    data.append(image)
    labels.append(0)


data = np.array(data)
labels = np.array(labels)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(data, labels,random_state=0)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
labels

In [ ]:
# Thực hiện tiền xử lý dữ liệu
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')

X_train/=255
X_test/=255

y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
X_train = X_train.reshape(-1,150*150*3)
X_test = X_test.reshape(-1,150*150*3)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
model = Sequential()
model.add(Dense(512,kernel_initializer='normal',activation='relu',input_shape=(67500,)))
model.add(Dense(512,activation='relu'))
model.add(Dropout(0.25))

model.add(Dense(256,kernel_initializer='normal',activation='relu'))
model.add(Dense(256,activation='relu'))
model.add(Dropout(0.25))

model.add(Dense(512,kernel_initializer='normal',activation='relu'))
model.add(Dense(512,activation='relu'))
model.add(Dropout(0.25))

model.add(Dense(256,kernel_initializer='normal',activation='relu'))
model.add(Dense(256,activation='relu'))
model.add(Dropout(0.25))

model.add(Dense(512,kernel_initializer='normal',activation='relu'))
model.add(Dense(512,activation='relu'))
model.add(Dropout(0.25))

model.add(Dense(512,kernel_initializer='normal',activation='relu'))
model.add(Dense(512,activation='relu'))
model.add(Dropout(0.25))



model.add(Dense(2,activation='softmax'))
model.summary()

In [ ]:
# Huận luyện mô hình 
model.compile(loss = 'mse', optimizer = RMSprop() , metrics = ['accuracy'])

In [ ]:
history = model.fit(X_train,y_train,batch_size=1000,epochs=200,verbose=1,validation_split=0.2,callbacks=[EarlyStopping(monitor='val_loss',patience=70)]) 

In [ ]:
# Đánh giá độ chính xác của mô hình
score = model.evaluate(X_test,y_test,verbose=0)
print('Sai số kiểm tra là: ',score[0])
print('Độ chính xác kiểm tra là: ',score[1])

In [ ]:
# vẽ lại quá trình học
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epochs')
plt.legend(['train','Validation'])
plt.show()

In [ ]:
from tensorflow.keras.models import load_model
model.save('Final.h5')
model_ANN = load_model('Final.h5')

In [ ]:
from tensorflow.keras.utils import load_img, img_to_array
import numpy as np
filename = "../input/datasetsfacedetect/True/37.png"

predict = ['Không phải Tuấn','Là Tuấn']
predict = np.array(predict)
img = load_img(filename,target_size=(150,150))
plt.imshow(img)

img = img_to_array(img)
img = img.reshape(1,150*150*3)
img = img.astype('float32')
img = img/255


result = np.argmax(model_ANN.predict(img),axis=-1)
predict[result]